In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmocean as cmo
from copy import copy

from matplotlib.colors import LightSource

In [ ]:
#name,xmin,xmax,ymin,ymax = 'PIG',-1.62e6,-1.55e6,-0.3e6,-0.2e6
#name,xmin,xmax,ymin,ymax = 'THW',-1.55e6,-1.47e6,-0.5e6,-0.4e6
#name,xmin,xmax,ymin,ymax = 'TOT',2.25e6,2.3e6,-1.05e6,-0.95e6
#name,xmin,xmax,ymin,ymax = 'DEN',2.4e6,2.55e6,-0.5e6,-0.3e6
#name,xmin,xmax,ymin,ymax = 'CRD',-1.53e6,-1.43e6,-0.72e6,-0.58e6
#name,xmin,xmax,ymin,ymax = 'ROS',-0.4e6,0e6,-0.6e6,-0.1e6
#name,xmin,xmax,ymin,ymax = 'RUT',-1.6e6,-1.2e6,0e6,0.4e6
#name,xmin,xmax,ymin,ymax = 'BUN',-1.0e6,-0.7e6,0.1e6,0.4e6
#name,xmin,xmax,ymin,ymax = 'ALL',-3.0e6,3.0e6,-3.0e6,3.0e6
name,xmin,xmax,ymin,ymax = 'AME',1.5e6,1.8e6,0.6e6,0.9e6


In [ ]:
ds = xr.open_dataset('../../../data/bedmachine/BedMachineAntarctica-v3.nc')
bmc3 = copy(ds.sel(x=slice(xmin,xmax),y=slice(ymax,ymin)))
ds.close()

ds = xr.open_dataset('../../../data/bedmap/bedmap3.nc')
bmp3 = copy(ds.sel(x=slice(xmin,xmax),y=slice(ymax,ymin)))
ds.close()

ds = xr.open_dataset('../../../data/bedmachine/NSIDC-0756_BedMachineAntarctica_19700101-20191001_V04.1.nc')
bmc4 = copy(ds.sel(x=slice(xmin,xmax),y=slice(ymax,ymin)))
ds.close()

In [ ]:
# Function to compute the hillshade
def compute_hillshade(elevation_data, azimuth=315, altitude=45):
    ls = LightSource(azdeg=azimuth, altdeg=altitude)
    hillshade = ls.hillshade(elevation_data, vert_exag=.1)
    return hillshade

In [ ]:
figsize = (7,4)
cbar_ratio = .05
ls = LightSource(azdeg=315, altdeg=45)
cmap = 'cmo.topo'

fig = plt.figure(figsize=figsize,constrained_layout=True)
spec = gridspec.GridSpec(2,3,figure=fig,height_ratios=[1,cbar_ratio],hspace=0.01,wspace=0.01,top=.99)
fax1 = fig.add_subplot(spec[0,0])
fax2 = fig.add_subplot(spec[0,1])
fax3 = fig.add_subplot(spec[0,2])
cax1 = fig.add_subplot(spec[1,:])


for fax,bm,suff,mlev in zip([fax1,fax2,fax3],[bmc3,bmp3,bmc4],['','_topography',''],[2.5,1.5,2.5]):
    hillshade = compute_hillshade(bm[f'bed{suff}'].values)
    fax.imshow(hillshade, extent=(bm.x.min(), bm.x.max(), bm.y.min(), bm.y.max()), cmap='gray', alpha=0.1,zorder=9)
    im = fax.pcolormesh(bm.x,bm.y,bm[f'bed{suff}'],cmap=cmap,vmin=-2000,vmax=2000,zorder=0)
    fax.contour(bm.x,bm.y,bm.mask,levels=[-2.5,mlev],colors='r',linewidths=.5,zorder=10)
    fax.contour(bm.x,bm.y,-bm[f'bed{suff}'],levels=np.arange(0,2000,100),colors='k',linewidths=.2)

    fax.set_aspect(1)
    fax.set_xticks([])
    fax.set_yticks([])

cbar = plt.colorbar(im,cax=cax1,orientation='horizontal',extend='both')
cbar.set_label('Bed [m]')

plt.savefig(f'../figures/beds_{name}.png',dpi=600)
